# 智瞳课堂 - YOLO 行为检测模型训练
使用数据增强 + 类别平衡训练，免费 GPU

In [ ]:
# 1. 安装依赖
!pip install ultralytics albumentations -q

import os, shutil, random
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
from google.colab import drive
from ultralytics import YOLO
import torch

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
drive.mount('/content/drive')

In [ ]:
# 2. 上传数据集（从本地上传 datasets/bvn 文件夹的 zip）
# 先在本地打包：把 datasets/bvn 文件夹压缩为 bvn.zip，然后上传
from google.colab import files

print('请上传 bvn.zip（包含 images/ 和 labels/ 文件夹）')
uploaded = files.upload()

!mkdir -p /content/datasets/bvn
!unzip -o bvn.zip -d /content/datasets/bvn/
!ls /content/datasets/bvn/

In [ ]:
# 3. 数据增强 + 平衡
import albumentations as A

CLASS_NAMES = ['hand-raising', 'reading', 'writing', 'using phone', 'bowing the head', 'leaning over the table']
NC = len(CLASS_NAMES)
SRC = '/content/datasets/bvn'
DST = '/content/datasets/bvn_balanced'

# 复制数据集
if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)

# 清理非文本文件（如 thumbs.db 等）
import glob as gl
for f in gl.glob(DST + '/labels/**/*.txt', recursive=True):
    try:
        open(f, encoding='utf-8').read(1)
    except:
        os.remove(f)
        print(f'删除异常文件: {f}')

# 增强管道
pipeline = A.Compose([
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        A.RandomGamma(gamma_limit=(80, 120), p=1.0),
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=20, val_shift_limit=15, p=1.0),
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
    ], p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=3, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.ImageCompression(quality_lower=70, quality_upper=95, p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.4, min_area=0.01))

# 统计
src_labels = Path(DST) / 'labels' / 'train'
src_images = Path(DST) / 'images' / 'train'
class_files = defaultdict(list)

for f in src_labels.glob('*.txt'):
    try:
        with open(f, encoding='utf-8') as fh:
            classes = set()
            for line in fh:
                parts = line.strip().split()
                if parts:
                    try:
                        cid = int(parts[0])
                        if 0 <= cid < NC: classes.add(cid)
                    except: pass
            for cid in classes:
                class_files[cid].append(f.stem)
    except:
        pass

print('原始分布:')
for cid in range(NC):
    print(f'  {CLASS_NAMES[cid]:25s}: {len(class_files[cid]):5d} 张')

# 增强到 60%
max_count = max(len(v) for v in class_files.values())
target = int(max_count * 0.6)
total_aug = 0

for cid in range(NC):
    files_list = class_files[cid]
    shortage = target - len(files_list)
    if shortage <= 0:
        continue
    print(f'增强 {CLASS_NAMES[cid]}: {shortage} 张...')
    generated = 0
    attempts = 0
    while generated < shortage and attempts < shortage * 3:
        src_file = random.choice(files_list)
        src_img = None
        for ext in ['.jpg', '.png', '.jpeg']:
            c = src_images / f'{src_file}{ext}'
            if c.exists(): src_img = c; break
        if src_img is None: attempts += 1; continue
        src_lbl = src_labels / f'{src_file}.txt'
        if not src_lbl.exists(): attempts += 1; continue

        try:
            image = cv2.imread(str(src_img))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            bboxes, clabels = [], []
            with open(src_lbl, encoding='utf-8') as fh:
                for line in fh:
                    p = line.strip().split()
                    if len(p) >= 5:
                        bboxes.append(list(map(float, p[1:5])))
                        clabels.append(int(p[0]))
            if not bboxes: attempts += 1; continue
            t = pipeline(image=image, bboxes=bboxes, class_labels=clabels)
            if not t['bboxes']: attempts += 1; continue

            aug_name = f'{src_file}_aug_{generated}'
            out_img = src_images / f'{aug_name}{src_img.suffix}'
            out_lbl = src_labels / f'{aug_name}.txt'
            cv2.imwrite(str(out_img), cv2.cvtColor(t['image'], cv2.COLOR_RGB2BGR))
            with open(out_lbl, 'w') as fh:
                for bb, cl in zip(t['bboxes'], t['class_labels']):
                    fh.write(f'{cl} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n')
            generated += 1
            total_aug += 1
        except: pass
        attempts += 1

print(f'\n共增强 {total_aug} 张')

# 显示增强后分布
print('\n增强后分布:')
class_files2 = defaultdict(list)
for f in src_labels.glob('*.txt'):
    try:
        with open(f, encoding='utf-8') as fh:
            classes = set()
            for line in fh:
                p = line.strip().split()
                if p:
                    try:
                        cid = int(p[0])
                        if 0 <= cid < NC: classes.add(cid)
                    except: pass
            for cid in classes:
                class_files2[cid].append(f.stem)
    except:
        pass
for cid in range(NC):
    print(f'  {CLASS_NAMES[cid]:25s}: {len(class_files2[cid]):5d} 张')

In [ ]:
# 5. 训练（每10轮自动保存到 Google Drive）
import shutil

model = YOLO('yolo11s.pt')

results = model.train(
    data=yaml_path,
    epochs=200,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/content/runs',
    name='train-augmented',
    exist_ok=True,
    patience=40,
    save_period=10,  # 每10轮保存checkpoint
    lr0=0.001,
    lrf=0.0001,
    cos_lr=True,
    close_mosaic=15,
    optimizer='AdamW',
    seed=42,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

# 训练完立即保存到 Google Drive（防丢失）
best_src = '/content/runs/train-augmented/weights/best.pt'
drive_dst = '/content/drive/MyDrive/class_behavior_best.pt'
if os.path.exists(best_src):
    shutil.copy(best_src, drive_dst)
    print(f'已保存到 Google Drive: {drive_dst}')
else:
    print('WARNING: best.pt 不存在，训练可能未完成')

print('训练完成!')

In [ ]:
# 5. 训练
model = YOLO('yolo11s.pt')

results = model.train(
    data=yaml_path,
    epochs=200,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/content/runs',
    name='train-augmented',
    exist_ok=True,
    patience=40,
    lr0=0.001,
    lrf=0.0001,
    cos_lr=True,
    close_mosaic=15,
    optimizer='AdamW',
    seed=42,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

print('训练完成!')

In [ ]:
# 7. 下载模型（从 Google Drive 或本地）
from google.colab import files
import os

# 先检查 Google Drive 里有没有
drive_model = '/content/drive/MyDrive/class_behavior_best.pt'
local_model = '/content/runs/train-augmented/weights/best.pt'

if os.path.exists(drive_model):
    print(f'Google Drive 中有模型: {drive_model}')
    files.download(drive_model)
elif os.path.exists(local_model):
    print(f'本地有模型: {local_model}')
    shutil.copy(local_model, drive_model)
    files.download(local_model)
else:
    # 找 checkpoint
    import glob
    checkpoints = glob.glob('/content/runs/train-augmented/weights/epoch*.pt')
    if checkpoints:
        latest = max(checkpoints, key=os.path.getmtime)
        print(f'使用最新 checkpoint: {latest}')
        shutil.copy(latest, drive_model)
        files.download(latest)
    else:
        print('没有找到任何模型文件，训练可能未完成')

print('下载完成后，把 best.pt 复制到 D:\\qiansai\\models\\class_behavior_best.pt')

In [ ]:
# 7. 下载模型
from google.colab import files

# 保存到 Google Drive
shutil.copy('/content/runs/train-augmented/weights/best.pt', '/content/drive/MyDrive/class_behavior_best.pt')
print('已保存到 Google Drive: class_behavior_best.pt')

# 也下载到本地
files.download('/content/runs/train-augmented/weights/best.pt')
print('下载完成后，把 best.pt 复制到 D:\\qiansai\\models\\class_behavior_best.pt')